In [1]:
import pandas as pd
import numpy as np
import joblib
import pickle
import gc
import time
from collections import defaultdict
from typing import List, Dict, Set, Optional, Tuple
from dataclasses import dataclass
from tqdm import tqdm
import logging
import lightgbm as lgb
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from datetime import timedelta
from sklearn.model_selection import train_test_split
import implicit
from scipy.sparse import coo_matrix
from implicit.evaluation import mean_average_precision_at_k
import optuna
from optuna.samplers import TPESampler
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
# =========================================================
# CONFIGURATION
# =========================================================

@dataclass
class Config:
    """Configuration du systeme"""
    N_ALS_CANDIDATES: int = 30
    N_POPULAR_CANDIDATES: int = 20
    N_REPURCHASE_CANDIDATES: int = 5
    TOP_K: int = 12
    
config = Config()

In [3]:
import pandas as pd
from datetime import timedelta

# -----------------------------
# 1. Chargement données
# -----------------------------
transactions = pd.read_csv('data/transactions_train.csv', parse_dates=['t_dat'])
customers = pd.read_csv('data/customers.csv')
articles = pd.read_csv('data/articles.csv')

print(f"Transactions: {transactions.shape}")
print(f"Customers: {customers.shape}")
print(f"Articles: {articles.shape}")

# -----------------------------
# 2. Split train/test (7 derniers jours)
# -----------------------------
# Date maximale du dataset
max_date = transactions['t_dat'].max()

# Fenêtre de test : dernière semaine
test_start = max_date - pd.Timedelta(days=6)  # dernière semaine
test_end = max_date

# Fenêtre d'entraînement : les 5 semaines avant la dernière semaine
train_start = test_start - pd.Timedelta(weeks=5)
transactions_train = transactions[(transactions['t_dat'] >= train_start) & (transactions['t_dat'] < test_start)]
transactions_test  = transactions[(transactions['t_dat'] >= test_start) & (transactions['t_dat'] <= test_end)]

print(f"Train: {transactions_train.shape}")
print(f"Test: {transactions_test.shape}")

# -----------------------------
# 3. Historique users
# -----------------------------
user_history_train = transactions_train.groupby('customer_id')['article_id'].apply(set).to_dict()
user_history_test = transactions_test.groupby('customer_id')['article_id'].apply(set).to_dict()

print(f"user_history_train: {len(user_history_train)} users")
print(f"user_history_test: {len(user_history_test)} users")

# -----------------------------
# 4. Mappings user/item
# -----------------------------
all_users_train = transactions_train['customer_id'].unique()
all_items_train = transactions_train['article_id'].unique()

user_map = {uid: idx for idx, uid in enumerate(all_users_train)}
item_map = {aid: idx for idx, aid in enumerate(all_items_train)}
ALL_ITEMS = list(all_items_train)

print(f"user_map: {len(user_map)} users")
print(f"item_map: {len(item_map)} items")
print(f"ALL_ITEMS: {len(ALL_ITEMS)} items")

# -----------------------------
# 5. Items populaires
# -----------------------------
top_popular_articles = transactions_train['article_id'].value_counts().head(100).index.tolist()
popular_items_idx = [item_map[aid] for aid in top_popular_articles if aid in item_map]

print(f"popular_items_idx: {len(popular_items_idx)} items")
print(f"Top 5: {[ALL_ITEMS[idx] for idx in popular_items_idx[:5]]}")


Transactions: (31788324, 5)
Customers: (1371980, 7)
Articles: (105542, 25)
Train: (1324934, 5)
Test: (240311, 5)
user_history_train: 274594 users
user_history_test: 68984 users
user_map: 274594 users
item_map: 30979 items
ALL_ITEMS: 30979 items
popular_items_idx: 100 items
Top 5: [751471001, 706016001, 918292001, 916468003, 915526001]


In [4]:
##Active
customers['Active'] = customers['Active'].fillna(0).astype(int)
# FN
customers['FN'] = customers['FN'].fillna(0).astype(int)
#age
customers['age'].unique()
customers['age'] = customers['age'].fillna(customers['age'].median())
#fashion news
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE')
customers['fashion_news_frequency'].unique()
#club_member_status
customers['club_member_status'].unique()
print(customers['club_member_status'].value_counts())
customers['club_member_status'] = customers['club_member_status'].fillna('UNKNOWN')
articles.drop(columns='detail_desc', inplace=True)
missing_articles = articles.isna().mean().sort_values(ascending=False)
missing_articles


club_member_status
ACTIVE        1272491
PRE-CREATE      92960
LEFT CLUB         467
Name: count, dtype: int64


article_id                      0.0
product_code                    0.0
garment_group_no                0.0
section_name                    0.0
section_no                      0.0
index_group_name                0.0
index_group_no                  0.0
index_name                      0.0
index_code                      0.0
department_name                 0.0
department_no                   0.0
perceived_colour_master_name    0.0
perceived_colour_master_id      0.0
perceived_colour_value_name     0.0
perceived_colour_value_id       0.0
colour_group_name               0.0
colour_group_code               0.0
graphical_appearance_name       0.0
graphical_appearance_no         0.0
product_group_name              0.0
product_type_name               0.0
product_type_no                 0.0
prod_name                       0.0
garment_group_name              0.0
dtype: float64

In [5]:
# =========================================================
# 5. FONCTION MAP@12
# =========================================================

def map_at_k_fast(preds: Dict, actuals: Dict, k: int = 12) -> float:
    """Calcule MAP@k vectorise"""
    
    scores = []
    for user_id, recs in preds.items():
        actual = actuals.get(user_id, set())
        if not actual:
            continue
        
        hits = np.array([item in actual for item in recs[:k]])
        if not hits.any():
            scores.append(0.0)
            continue
        
        cumsum_hits = np.cumsum(hits)
        precisions = cumsum_hits * hits / np.arange(1, len(hits) + 1)
        scores.append(precisions.sum() / min(len(actual), k))
    
    return float(np.mean(scores)) if scores else 0.0

In [6]:
def train_als_weighted(transactions_train, user_map, item_map,
                       factors=50, iterations=15, top_n=100, decay_days=30):
    """
    Entraine ALS avec pondération des interactions selon la récence
    et génère cache top-N items par utilisateur.
    """
    # Copier et mapper user/item
    df = transactions_train.copy()
    df['user_idx'] = df['customer_id'].map(user_map)
    df['item_idx'] = df['article_id'].map(item_map)
    df = df.dropna(subset=['user_idx','item_idx'])

    # Calcul pondération par récence
    max_date = df['t_dat'].max()
    df['days_ago'] = (max_date - df['t_dat']).dt.days
    df['weight'] = np.exp(-df['days_ago'] / decay_days).astype(np.float32)

    # Matrice sparse pondérée
    interaction_matrix = csr_matrix(
        (df['weight'], (df['user_idx'].astype(int), df['item_idx'].astype(int))),
        shape=(len(user_map), len(item_map))
    )

    # Entrainer ALS
    model = AlternatingLeastSquares(factors=factors, iterations=iterations,
                                    regularization=0.01, random_state=42)
    model.fit(interaction_matrix)

    # Générer cache top-N items par utilisateur
    als_cache = {}
    for uid in tqdm(range(len(user_map)), desc="ALS cache"):
        item_indices, scores = model.recommend(uid, interaction_matrix[uid],
                                              N=top_n, filter_already_liked_items=False)
        als_cache[uid] = {int(i): float(s) for i, s in zip(item_indices, scores)}

    return als_cache, model

# -----------------------------
# Entraînement
# -----------------------------
als_cache, als_model = train_als_weighted(transactions_train, user_map, item_map,
                                          factors=50, iterations=15, top_n=100, decay_days=30)

# Sauvegarde cache
with open('als_cache_weighted.pkl', 'wb') as f:
    pickle.dump(als_cache, f)

print(f"ALS cache pondéré généré pour {len(als_cache)} users")

C:\Users\ajarr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

ALS cache: 100%|██████████| 274594/274594 [04:35<00:00, 996.50it/s] 


ALS cache pondéré généré pour 274594 users


In [7]:
from collections import defaultdict

def build_actuals(transactions_test, user_map, item_map):
    """
    Construit le ground truth : user_idx -> set(item_idx)
    """
    actuals = defaultdict(set)

    df = transactions_test.copy()
    df['user_idx'] = df['customer_id'].map(user_map)
    df['item_idx'] = df['article_id'].map(item_map)
    df = df.dropna(subset=['user_idx', 'item_idx'])

    for row in df.itertuples():
        actuals[int(row.user_idx)].add(int(row.item_idx))

    return actuals
def build_preds_from_als_cache(als_cache):
    """
    Convertit le cache ALS en format MAP :
    user_idx -> [item_idx1, item_idx2, ...]
    """
    preds = {}

    for user_idx, item_scores in als_cache.items():
        # Trier par score décroissant
        ranked_items = sorted(
            item_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
        preds[user_idx] = [item for item, _ in ranked_items]

    return preds
# Construire actuals et preds
actuals = build_actuals(transactions_test, user_map, item_map)
preds = build_preds_from_als_cache(als_cache)

# Calcul MAP@12
map12 = map_at_k_fast(preds, actuals, k=12)

print(f" MAP@12 ALS pondéré : {map12:.5f}")


 MAP@12 ALS pondéré : 0.01752


In [8]:
# =========================================================
# Fonction batch
# =========================================================
def batch(iterable, batch_size):
    """Divise un iterable en batches de taille batch_size"""
    length = len(iterable)
    for i in range(0, length, batch_size):
        yield iterable[i:min(i + batch_size, length)]

# =========================================================
# Features utilisateurs
# =========================================================
def create_user_features(customers):
    user_feat = customers[['customer_id','age','club_member_status','fashion_news_frequency']].copy()
    user_feat['club_member'] = (user_feat['club_member_status']=='ACTIVE').astype(int)
    user_feat['news_regular'] = (user_feat['fashion_news_frequency']=='Regularly').astype(int)
    user_feat['age_group'] = pd.cut(user_feat['age'], bins=[0,25,35,45,55,100], labels=[0,1,2,3,4]).astype(float)
    return user_feat.set_index('customer_id')[['age_group','club_member','news_regular']]

# =========================================================
# Features articles
# =========================================================
def create_item_features(articles):
    item_feat = articles[['article_id','product_type_no','product_group_name','colour_group_code','index_group_no']].copy()
    return item_feat.set_index('article_id')

# =========================================================
# Features interaction - VERSION OPTIMISÉE MÉMOIRE
# =========================================================
def create_interaction_features(transactions, articles):
    """Version optimisée pour éviter MemoryError"""
    print("  Calcul ui_hist...")
    
    # Réduire la taille avant groupby
    trans_light = transactions[['customer_id', 'article_id', 't_dat', 'price']].copy()
    
    # Groupby avec des types optimisés
    ui_hist = trans_light.groupby(['customer_id','article_id'], observed=True).agg({
        't_dat': 'max',
        'price': ['mean', 'count']
    })
    
    # Flatten columns
    ui_hist.columns = ['last_purchase_ui', 'avg_price_ui', 'n_purchases_ui']
    ui_hist = ui_hist.reset_index()
    
    # Calculer recency en jours
    max_date = transactions['t_dat'].max()
    ui_hist['recency_ui_days'] = (max_date - ui_hist['last_purchase_ui']).dt.days.astype('int16')
    
    # Optimiser types
    ui_hist['n_purchases_ui'] = ui_hist['n_purchases_ui'].astype('int16')
    ui_hist['avg_price_ui'] = ui_hist['avg_price_ui'].astype('float32')
    ui_hist.drop('last_purchase_ui', axis=1, inplace=True)
    
    del trans_light
    gc.collect()
    
    print("  Calcul category affinity...")
    # Catégorie - version optimisée
    cat_aff = transactions[['customer_id', 'article_id']].merge(
        articles[['article_id','product_type_no']], 
        on='article_id',
        how='left'
    ).groupby(['customer_id','product_type_no'], observed=True).size().reset_index(name='category_purchases')
    cat_aff['category_purchases'] = cat_aff['category_purchases'].astype('int16')
    
    print("  Calcul color affinity...")
    # Couleur - version optimisée
    color_aff = transactions[['customer_id', 'article_id']].merge(
        articles[['article_id','colour_group_code']], 
        on='article_id',
        how='left'
    ).groupby(['customer_id','colour_group_code'], observed=True).size().reset_index(name='color_purchases')
    color_aff['color_purchases'] = color_aff['color_purchases'].astype('int16')
    
    gc.collect()
    return {'ui_hist': ui_hist, 'cat_aff': cat_aff, 'color_aff': color_aff}

In [9]:
# =========================================================
# Classe Candidate Generator
# =========================================================
class CandidateGenerator:
    """Generation candidats: ALS + Popular + Repurchase"""
    
    def __init__(self,
                 als_cache: Dict,
                 popular_items_idx: List[int],
                 user_history_train: Dict,
                 item_map: Dict,
                 ALL_ITEMS: List,
                 n_als: int = 30,
                 n_popular: int = 20,
                 n_repurchase: int = 5):
        
        self.als_cache = als_cache
        self.popular_items_idx = popular_items_idx[:n_popular]
        self.user_history_train = user_history_train
        self.item_map = item_map
        self.ALL_ITEMS = ALL_ITEMS
        self.n_als = n_als
        self.n_repurchase = n_repurchase
    
    def generate(self, user_id: str, user_idx: Optional[int]) -> Dict:
        """
        Genere candidats pour un user.
        Retourne: {item_idx: {'als_score': float, 'popular': int, 'repurchase': int}}
        """
        candidates = {}
        
        # 1. ALS top-N
        if user_idx is not None and user_idx in self.als_cache:
            als_items = self.als_cache[user_idx]
            sorted_als = sorted(als_items.items(), key=lambda x: x[1], reverse=True)
            
            for item_idx, score in sorted_als[:self.n_als]:
                candidates[item_idx] = {
                    'als_score': float(score),
                    'popular': 0,
                    'repurchase': 0
                }
        
        # 2. Popular top-N
        for item_idx in self.popular_items_idx:
            if item_idx not in candidates:
                candidates[item_idx] = {
                    'als_score': 0.0,
                    'popular': 1,
                    'repurchase': 0
                }
            else:
                candidates[item_idx]['popular'] = 1
        
        # 3. Repurchase top-N
        user_hist = list(self.user_history_train.get(user_id, set()))
        for aid in user_hist[:self.n_repurchase]:
            item_idx = self.item_map.get(aid)
            if item_idx is not None:
                if item_idx not in candidates:
                    candidates[item_idx] = {
                        'als_score': 0.0,
                        'popular': 0,
                        'repurchase': 1
                    }
                else:
                    candidates[item_idx]['repurchase'] = 1
        
        return candidates

In [10]:
# =========================================================
# Fonction construction dataset
# =========================================================
def build_dataset_candidates(users, candidate_gen, user_features, item_features,
                             interaction_features, user_history_test, selected_features):
    """Construit le dataset pour un batch d'utilisateurs"""
    rows = []
    
    # Pré-indexer pour accès rapide
    ui_hist_indexed = None
    cat_aff_dict = None
    color_aff_dict = None
    
    if 'ui_hist' in interaction_features:
        ui_hist_indexed = interaction_features['ui_hist'].set_index(['customer_id', 'article_id'])
    
    if 'cat_aff' in interaction_features:
        cat_aff_dict = interaction_features['cat_aff'].set_index(['customer_id', 'product_type_no'])['category_purchases'].to_dict()
    
    if 'color_aff' in interaction_features:
        color_aff_dict = interaction_features['color_aff'].set_index(['customer_id', 'colour_group_code'])['color_purchases'].to_dict()
    
    for user_id in tqdm(users, desc="Building candidates", leave=False):
        if user_id not in user_features.index:
            continue
            
        user_idx = user_map.get(user_id)
        candidates = candidate_gen.generate(user_id, user_idx)
        
        u_feat = user_features.loc[user_id]
        bought_test = user_history_test.get(user_id, set())

        for item_idx, info in candidates.items():
            article_id = ALL_ITEMS[item_idx]
            if article_id not in item_features.index:
                continue
            i_feat = item_features.loc[article_id]

            row = {
                'user_id': user_id,
                'item_id': article_id,
                'label': int(article_id in bought_test),
            }

            # Features candidats: ALS / Popular / Repurchase
            for f in ['als_score', 'popular', 'repurchase']:
                if f in selected_features:
                    row[f] = info.get(f, 0)

            # User features
            for f in selected_features:
                if f.startswith('user_'):
                    col_name = f[5:]  # Enlever 'user_'
                    row[f] = u_feat.get(col_name, 0) if col_name in u_feat.index else 0

            # Item features
            for f in selected_features:
                if f.startswith('item_'):
                    col_name = f[5:]  # Enlever 'item_'
                    row[f] = i_feat.get(col_name, 0) if col_name in i_feat.index else 0

            # Interaction features user-item
            if ui_hist_indexed is not None:
                if 'n_purchases_ui' in selected_features or 'recency_ui' in selected_features or 'avg_price_ui' in selected_features:
                    try:
                        ui_data = ui_hist_indexed.loc[(user_id, article_id)]
                        if 'n_purchases_ui' in selected_features:
                            row['n_purchases_ui'] = ui_data.get('n_purchases_ui', 0)
                        if 'recency_ui' in selected_features:
                            row['recency_ui'] = ui_data.get('recency_ui_days', 999)
                        if 'avg_price_ui' in selected_features:
                            row['avg_price_ui'] = ui_data.get('avg_price_ui', 0)
                    except KeyError:
                        if 'n_purchases_ui' in selected_features:
                            row['n_purchases_ui'] = 0
                        if 'recency_ui' in selected_features:
                            row['recency_ui'] = 999
                        if 'avg_price_ui' in selected_features:
                            row['avg_price_ui'] = 0
            
            # Category affinity
            if 'category_affinity' in selected_features and cat_aff_dict is not None:
                row['category_affinity'] = cat_aff_dict.get((user_id, i_feat.get('product_type_no', 0)), 0)
            
            # Color affinity
            if 'color_affinity' in selected_features and color_aff_dict is not None:
                row['color_affinity'] = color_aff_dict.get((user_id, i_feat.get('colour_group_code', 0)), 0)

            rows.append(row)
    
    return pd.DataFrame(rows)

In [ ]:
# =========================================================
# PIPELINE PRINCIPAL
# =========================================================

print("="*60)
print("PIPELINE LIGHTGBM RANKER")
print("="*60)

# 1. Définir les features à utiliser
print("\n1. Définition des features...")
selected_features = [
    # Candidat features
    'als_score', 'popular', 'repurchase',
    
    # User features
    'user_age_group', 'user_club_member', 'user_news_regular',
    
    # Item features
    'item_product_type', 'item_product_group', 'item_colour_group', 'item_index_group',
    
    # Interaction features
    'n_purchases_ui', 'recency_ui', 'avg_price_ui',
    'category_affinity', 'color_affinity'
]
print(f"Features sélectionnées: {len(selected_features)}")

# 2. Conversion types
print("\n2. Conversion des types...")
transactions_train['customer_id'] = transactions_train['customer_id'].astype('category')
transactions_train['article_id'] = transactions_train['article_id'].astype('category')
transactions_train['price'] = transactions_train['price'].astype('float32')

customers['customer_id'] = customers['customer_id'].astype('category')
articles['article_id'] = articles['article_id'].astype('category')

# 3. Créer features
print("\n3. Création des features...")
user_features = create_user_features(customers)
item_features = create_item_features(articles)
interaction_features = create_interaction_features(transactions_train, articles)

# Optimiser types interaction_features
if 'ui_hist' in interaction_features:
    interaction_features['ui_hist']['customer_id'] = interaction_features['ui_hist']['customer_id'].astype('category')
    interaction_features['ui_hist']['article_id'] = interaction_features['ui_hist']['article_id'].astype('category')

print(f"User features: {user_features.shape}")
print(f"Item features: {item_features.shape}")

# 4. Initialiser Candidate Generator
print("\n4. Initialisation du générateur de candidats...")
candidate_gen = CandidateGenerator(
    als_cache=als_cache,
    popular_items_idx=popular_items_idx,
    user_history_train=user_history_train,
    item_map=item_map,
    ALL_ITEMS=ALL_ITEMS,
    n_als=30,
    n_popular=20,
    n_repurchase=5
)

# 5. Construire dataset par batches
print("\n5. Construction du dataset par batches...")
users = list(user_history_train.keys())
chunks = []

for batch_users in tqdm(list(batch(users, 1000)), desc="Build batches"):
    X_batch = build_dataset_candidates(
        batch_users,
        candidate_gen,
        user_features,
        item_features,
        interaction_features,
        user_history_test,
        selected_features
    )
    
    # Convertir colonnes catégorielles
    cat_cols = ['user_id', 'item_id', 'item_product_group', 'item_product_type', 
                'item_colour_group', 'item_index_group']
    for col in cat_cols:
        if col in X_batch.columns and X_batch[col].dtype == 'object':
            X_batch[col] = X_batch[col].astype('category')
    
    chunks.append(X_batch)
    del X_batch
    gc.collect()

print("\n6. Concatenation des batches...")
X_df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

print(f"Dataset final: {X_df.shape[0]} lignes, {X_df.shape[1]} colonnes")
print(f"Distribution labels:\n{X_df['label'].value_counts()}")
print(f"Taux positif: {X_df['label'].mean():.4f}")

# 7. Préparer données pour LightGBM
print("\n7. Préparation pour LightGBM...")
y = X_df['label'].values
X = X_df[selected_features].copy()
group = X_df.groupby('user_id', observed=True).size().values

print(f"X shape: {X.shape}")
print(f"Nombre de groupes: {len(group)}")
print(f"Taille moyenne groupe: {group.mean():.1f}")

# 8. Entraîner LightGBM Ranker
print("\n8. Entraînement LightGBM Ranker...")
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='map',
    eval_at=[12],
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=8,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    force_row_wise=True,
    verbose=-1
)

ranker.fit(X, y, group=group)
print("✓ Entraînement terminé!")

# 9. Prédictions et évaluation
print("\n9. Génération des prédictions...")
X_df['pred_score'] = ranker.predict(X)

preds_dict = {}
for user_id, group_df in tqdm(X_df.groupby('user_id', observed=True), desc="Top-12 par user"):
    top_items = group_df.nlargest(12, 'pred_score')['item_id'].tolist()
    preds_dict[user_id] = top_items

map12 = map_at_k_fast(preds_dict, user_history_test, k=12)

print("\n" + "="*60)
print(f"MAP@12 = {map12:.6f}")
print("="*60)

# 10. Feature importance
print("\n10. Feature importance (top 15):")
importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': ranker.feature_importances_
}).sort_values('importance', ascending=False)
print(importance_df.head(15).to_string(index=False))

# 11. Nettoyage
print("\n11. Nettoyage mémoire...")
del X_df, X, y, group
gc.collect()
print("✓ Terminé!")

PIPELINE LIGHTGBM RANKER

1. Définition des features...
Features sélectionnées: 15

2. Conversion des types...


C:\Users\ajarr\AppData\Local\Temp\ipykernel_43648\1883451040.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transactions_train['customer_id'] = transactions_train['customer_id'].astype('category')
C:\Users\ajarr\AppData\Local\Temp\ipykernel_43648\1883451040.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transactions_train['article_id'] = transactions_train['article_id'].astype('category')
C:\Users\ajarr\AppData\Local\Temp\ipykernel_43648\1883451040.py:31: SettingWithCopyWarning: 
A value is tryi


3. Création des features...
  Calcul ui_hist...
  Calcul category affinity...
  Calcul color affinity...
User features: (1371980, 3)
Item features: (105542, 4)

4. Initialisation du générateur de candidats...

5. Construction du dataset par batches...


Build batches:  28%|██▊       | 76/275 [55:20<2:05:43, 37.91s/it]

In [ ]:
import pandas as pd
from tqdm import tqdm

# Charger le sample_submission
sample_submission = pd.read_csv("data/sample_submission.csv")

# Liste pour stocker les prédictions
predictions = []

# Statistiques pour suivi
stats = {'cached': 0, 'fallback_popular': 0, 'errors': 0}

# Boucle sur tous les utilisateurs du sample_submission
for user_id in tqdm(sample_submission['customer_id'], desc="Génération recommendations"):
    try:
        # 1. Si on a déjà des prédictions LGBM
        if user_id in preds_dict:
            items = preds_dict[user_id]
            stats['cached'] += 1

        # 2. Sinon fallback → top populaire global
        else:
            items = popular_items_idx[:12]
            stats['fallback_popular'] += 1

        # Format Kaggle → 10 caractères
        predictions.append(' '.join(str(i).zfill(10) for i in items))

    except Exception as e:
        # En cas d'erreur, fallback sur top global
        predictions.append(' '.join(str(i).zfill(10) for i in popular_items_idx[:12]))
        stats['errors'] += 1
        print(f"Erreur sur user {user_id}: {e}")

# Ajouter la colonne "prediction" au sample_submission
sample_submission['prediction'] = predictions

# Sauvegarder le fichier CSV final
sample_submission.to_csv("submission2.csv", index=False)
print(" Submission générée: 'submission2.csv'")

# Afficher statistiques
print("Statistiques génération:")
print(stats)


Génération recommendations: 100%|██████████| 1371980/1371980 [00:03<00:00, 380738.10it/s]


 Submission générée: 'submission2.csv'
Statistiques génération:
{'cached': 274594, 'fallback_popular': 1097386, 'errors': 0}


In [ ]:
import pandas as pd

#  Charger ta soumission existante
sub = pd.read_csv("submission1.csv")

# Vérification rapide
print(sub.head())

#  Fonction pour corriger les article_id
def fix_prediction(pred):
    # pred = string "568601006 795440001 ..."
    return " ".join([str(a).zfill(10) for a in pred.split()])

#  Appliquer la correction
sub["prediction"] = sub["prediction"].apply(fix_prediction)

#  Vérification finale
print("Tous les article_id font 10 caractères ?",
      sub["prediction"].str.split().apply(lambda x: all(len(a)==10 for a in x)).all())

#  Sauvegarder le nouveau fichier
sub.to_csv("submission1_fixed.csv", index=False)

print(" Fichier sauvegardé : submission1_fixed.csv")


                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  568601006 795440001 568597006 795440003 615141...  
1  351484002 599580055 673677002 759871002 723529...  
2  351484002 723529001 609719001 458543001 699080...  
3  730683001 564786001 757303012 720125007 678687...  
4  698286003 692721005 791587001 707704003 730683...  
Tous les article_id font 10 caractères ? True
✅ Fichier sauvegardé : submission1_fixed.csv


2ème approche plus de featues

In [ ]:
import pandas as pd
import numpy as np
import gc
from collections import defaultdict
from tqdm import tqdm
from datetime import timedelta
from sklearn.feature_selection import mutual_info_classif
import lightgbm as lgb



# -----------------------------
# Features utilisateurs
# -----------------------------
def create_user_features(transactions, customers):
    transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
    user_stats = transactions.groupby('customer_id').agg({
        'article_id': 'count',
        'price': ['sum','mean','std'],
        't_dat': ['min','max']
    })
    user_stats.columns = ['n_purchases','total_spent','avg_price','std_price','first_purchase','last_purchase']
    max_date = transactions['t_dat'].max()
    user_stats['recency_days'] = (max_date - user_stats['last_purchase']).dt.days
    user_stats['lifetime_days'] = (user_stats['last_purchase'] - user_stats['first_purchase']).dt.days + 1
    user_stats['purchase_frequency'] = user_stats['n_purchases'] / user_stats['lifetime_days']
    user_diversity = transactions.groupby('customer_id')['article_id'].nunique()
    user_stats['n_unique_items'] = user_diversity
    user_stats['diversity_rate'] = user_diversity / user_stats['n_purchases']
    # Merge customers
    user_stats = user_stats.merge(
        customers[['customer_id','age','club_member_status','fashion_news_frequency']],
        left_index=True, right_on='customer_id', how='left'
    )
    user_stats['age_group'] = pd.cut(
        user_stats['age'],
        bins=[0,25,35,45,55,100],
        labels=[0,1,2,3,4],
        right=False
    ).astype(float)
    user_stats['club_member'] = (user_stats['club_member_status']=='ACTIVE').astype(int)
    user_stats['news_regular'] = (user_stats['fashion_news_frequency']=='Regularly').astype(int)
    return user_stats.fillna(0).set_index('customer_id')

# -----------------------------
# Features articles
# -----------------------------
def create_item_features(transactions, articles):
    transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
    item_stats = transactions.groupby('article_id').agg({
        'customer_id':['count','nunique'],
        'price':['mean','std']
    })
    item_stats.columns = ['n_sales','n_unique_users','avg_price','std_price']
    max_date = transactions['t_dat'].max()
    for w in [1,2,4]:
        recent = transactions[transactions['t_dat'] > (max_date - timedelta(weeks=w))]
        recent_pop = recent.groupby('article_id').size()
        item_stats[f'sales_last_{w}w'] = item_stats.index.to_series().map(recent_pop).fillna(0)
    item_stats['trend_1w'] = item_stats['sales_last_1w'] / (item_stats['n_sales'] + 1)
    item_stats['trend_4w'] = item_stats['sales_last_4w'] / (item_stats['n_sales'] + 1)
    item_stats['repurchase_rate'] = item_stats['n_sales'] / item_stats['n_unique_users']
    item_stats = item_stats.merge(
        articles[['article_id','product_type_no','product_group_name','colour_group_code','index_group_no']],
        left_index=True, right_on='article_id', how='left'
    )
    return item_stats.fillna(0).set_index('article_id')

# -----------------------------
# Features interaction (optionnel)
# -----------------------------
def create_interaction_features(transactions, articles):
    return {}  # on ne calcule que pour les candidats pour économie mémoire

# -----------------------------
# Build dataset uniquement pour les candidats
# -----------------------------
def build_dataset_candidates(users, candidate_gen, user_features, item_features,
                             interaction_features, user_history_test, selected_features=None):
    rows = []
    ui_hist_dict = {}
    cat_aff_dict = {}
    color_aff_dict = {}
    for user_id in tqdm(users, desc="Build dataset candidates"):
        if user_id not in user_features.index:
            continue
        u_feat = user_features.loc[user_id]
        user_idx = user_map.get(user_id, None)
        candidates = candidate_gen.generate(user_id, user_idx)
        bought_test = user_history_test.get(user_id, set())
        bought_train = candidate_gen.user_history_train.get(user_id, set())
        for item_idx, info in candidates.items():
            article_id = candidate_gen.ALL_ITEMS[item_idx]
            if article_id not in item_features.index:
                continue
            i_feat = item_features.loc[article_id]
            row = {'user_id': user_id, 'item_id': article_id, 'label': int(article_id in bought_test)}
            # Features candidats
            for f in ['als_score','popular','repurchase']:
                if selected_features is None or f in selected_features:
                    row[f] = info.get(f,0)
            # Features user / item
            for feat_name in ['recency_days','purchase_frequency','n_purchases','diversity_rate','avg_price',
                              'age_group','club_member','news_regular','n_sales','trend_1w','trend_4w',
                              'repurchase_rate','avg_price','sales_last_1w','sales_last_2w','sales_last_4w',
                              'product_type_no']:
                if selected_features is None or feat_name in selected_features:
                    if feat_name in u_feat:
                        row[f'user_{feat_name}'] = u_feat[feat_name]
                    elif feat_name in i_feat:
                        row[f'item_{feat_name}'] = i_feat[feat_name]
            rows.append(row)
    return pd.DataFrame(rows)

# -----------------------------
# Sélection features Mutual Information
# -----------------------------
def select_best_features(X, y, threshold=0.001):
    mi_scores = mutual_info_classif(X, y, random_state=42, n_jobs=-1)
    mi_df = pd.DataFrame({'feature': X.columns,'mi_score':mi_scores}).sort_values('mi_score',ascending=False)
    print("\nTop 15 features (MI):")
    print(mi_df.head(15).to_string(index=False))
    selected = mi_df[mi_df['mi_score']>threshold]['feature'].tolist()
    print(f"\nFeatures retenues: {len(selected)}/{len(X.columns)}")
    return selected, mi_df

# =========================================================
# PIPELINE HYBRIDE (modèle2)
# =========================================================
print("="*60)
print("PIPELINE HYBRIDE OPTIMISÉ (ALS + LightGBM, modèle2)")
print("="*60)

# Features utilisateurs et items
user_features = create_user_features(transactions_train, customers)
item_features = create_item_features(transactions_train, articles)
interaction_features = create_interaction_features(transactions_train, articles)

# Générateur de candidats
candidate_gen = CandidateGenerator(
    als_cache=als_cache,
    popular_items_idx=popular_items_idx,
    user_history_train=user_history_train,
    item_map=item_map,
    ALL_ITEMS=ALL_ITEMS,
    n_als=20,
    n_popular=15,
    n_repurchase=3
)

# Construire dataset batch par batch
users = list(user_history_train.keys())
chunks = []
for batch_users in tqdm(list(batch(users, 1000)), desc="Build batches"):
    X_batch = build_dataset_candidates(batch_users, candidate_gen, user_features,
                                       item_features, interaction_features,
                                       user_history_test, selected_features=None)
    # Colonnes catégorielles
    cat_cols = ['user_id','item_id']
    for col in cat_cols:
        if col in X_batch.columns and X_batch[col].dtype=='object':
            X_batch[col] = X_batch[col].astype('category')
    chunks.append(X_batch)
    del X_batch
    gc.collect()

X_df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()
print(f"Dataset final: {X_df.shape[0]} lignes, {X_df.shape[1]} colonnes")

# Sélection features par MI
y = X_df['label'].values
X_all = X_df.drop(columns=['user_id','item_id','label'])
selected_features, mi_df = select_best_features(X_all, y, threshold=0.001)
print(f"Features retenues pour le Ranker modèle2: {selected_features}")

# Préparer données LightGBM
X = X_df[selected_features].copy()
group = X_df.groupby('user_id', observed=True).size().values

# Entraînement LightGBM
ranker_model2 = lgb.LGBMRanker(
    objective='lambdarank',
    metric='map',
    eval_at=[12],
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=8,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    force_row_wise=True,
    verbose=-1
)
ranker_model2.fit(X, y, group=group)

# Générer prédictions top-12
X_df['pred_score'] = ranker_model2.predict(X)
preds_dict_model2 = {user_id: df.nlargest(12,'pred_score')['item_id'].tolist()
                     for user_id, df in X_df.groupby('user_id', observed=True)}

map12_model2 = map_at_k_fast(preds_dict_model2, user_history_test, k=12)
print(f"MAP@12 modèle2 = {map12_model2:.6f}")

# Feature importance
importance_df_model2 = pd.DataFrame({
    'feature': selected_features,
    'importance': ranker_model2.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 15 features modèle2:")
print(importance_df_model2.head(15).to_string(index=False))

# Nettoyage
del X_df, X, y, group
gc.collect()
print("✓ Pipeline modèle2 terminé et mémoire libérée!")


PIPELINE HYBRIDE OPTIMISÉ (ALS + LightGBM, modèle2)


C:\Users\ajarr\AppData\Local\Temp\ipykernel_15156\2468161888.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
C:\Users\ajarr\AppData\Local\Temp\ipykernel_15156\2468161888.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  user_stats = transactions.groupby('customer_id').agg({
C:\Users\ajarr\AppData\Local\Temp\ipykernel_15156\2468161888.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain cur

Dataset final: 9811002 lignes, 22 colonnes


MemoryError: Unable to allocate 299. MiB for an array with shape (9803034, 4) and data type float64

In [ ]:
# -------------------------
# 1. SAMPLING STRATIFIÉ POUR MI (ROBUSTE)
# -------------------------
sample_size = min(1_000_000, len(y))
target_pos_ratio = 0.2
rng = np.random.default_rng(42)

y_arr = np.asarray(y, dtype=np.uint8)

pos_idx = np.flatnonzero(y_arr)
neg_idx = np.flatnonzero(1 - y_arr)

n_pos = min(len(pos_idx), int(sample_size * target_pos_ratio))
n_neg = min(len(neg_idx), sample_size - n_pos)

# Ajustement sécurité
if n_pos + n_neg < sample_size:
    n_pos = min(len(pos_idx), sample_size - n_neg)

idx = np.empty(n_pos + n_neg, dtype=np.int32)
idx[:n_pos] = rng.choice(pos_idx, n_pos, replace=False)
idx[n_pos:] = rng.choice(neg_idx, n_neg, replace=False)
rng.shuffle(idx)

X_mi = X_all.iloc[idx]
y_mi = y_arr[idx]

# -------------------------
# 2. SÉLECTION DES FEATURES PAR MI
# -------------------------
selected_features, mi_df = select_best_features(
    X_mi, y_mi, threshold=0.001
)

# Sécurité si MI trop stricte
if not selected_features:
    print("⚠️ Aucune feature sélectionnée par MI, fallback sur toutes les features.")
    selected_features = X_all.columns.tolist()

del X_mi, y_mi, idx
gc.collect()

# -------------------------
# 3. PRÉPARATION DES DONNÉES LIGHTGBM
# -------------------------
X = X_df[selected_features].copy()

# Downcast mémoire
for col in X.columns:
    if X[col].dtype == 'float64':
        X[col] = X[col].astype('float32')
    elif X[col].dtype == 'int64':
        X[col] = X[col].astype('int32')


NameError: name 'y' is not defined

In [ ]:
group = (
    X_df.groupby('user_id', observed=True)
    .size()
    .values
)
# -------------------------
# Vérification du pourcentage de positifs
# -------------------------
n_total = len(y)
n_pos = (y == 1).sum()
percent_pos = 100 * n_pos / n_total
print(f"Total lignes: {n_total}, positifs: {n_pos} ({percent_pos:.2f}%)")

# -------------------------
# 4. ENTRAÎNEMENT LIGHTGBM RANKER
# -------------------------
ranker_model2 = lgb.LGBMRanker(
    objective='lambdarank',
    metric='map',
    eval_at=[12],
    boosting_type='gbdt',
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=8,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    force_row_wise=True,
    verbose=-1
)

ranker_model2.fit(X, y, group=group)

# -------------------------
# 5. PRÉDICTION & TOP-12
# -------------------------
X_df['pred_score'] = ranker_model2.predict(
    X, num_iteration=ranker_model2.best_iteration_
)

X_df.sort_values(
    ['user_id', 'pred_score'],
    ascending=[True, False],
    inplace=True
)

preds_dict_model2 = (
    X_df.groupby('user_id', observed=True)['item_id']
    .head(12)
    .groupby(X_df['user_id'])
    .apply(list)
    .to_dict()
)

# -------------------------
# 6. ÉVALUATION MAP@12
# -------------------------
map12_model2 = map_at_k_fast(preds_dict_model2, user_history_test, k=12)
print(f"\n MAP@12 modèle2 = {map12_model2:.6f}")

# -------------------------
# 7. FEATURE IMPORTANCE
# -------------------------
importance_df_model2 = (
    pd.DataFrame({
        'feature': selected_features,
        'importance': ranker_model2.feature_importances_
    })
    .sort_values('importance', ascending=False)
)

print("\n🔝 Top 15 features modèle2 :")
print(importance_df_model2.head(15).to_string(index=False))

# -------------------------
# 8. NETTOYAGE MÉMOIRE
# -------------------------
del X, X_all, X_df, y, group
gc.collect()

print("\n✓ Pipeline modèle2 terminé – mémoire libérée.")

NameError: name 'X_df' is not defined

In [ ]:
# =========================================================
# 8. OPTIMISATION HYPERPARAM LIGHTGBM AVEC OPTUNA (avec print)
# =========================================================
import optuna
from optuna.samplers import TPESampler
import joblib
import time

def optimize_with_optuna(
    transactions_train,
    transactions_test,
    user_features_base,
    item_features_base,
    interaction_features,
    selected_features,
    user_history_train,
    user_history_test,
    als_cache,
    popular_items_idx,
    user_map,
    item_map,
    ALL_ITEMS,
    n_trials: int = 30
):
    """
    Optimise les hyperparamètres LightGBM Ranker avec Optuna.
    Retourne le study, le modèle final et le meilleur MAP@12.
    """
    print("=" * 70)
    print(f"OPTIMISATION OPTUNA - {n_trials} TRIALS")
    print("=" * 70)
    
    global_data = {
        'transactions_train': transactions_train,
        'transactions_test': transactions_test,
        'user_features_base': user_features_base,
        'item_features_base': item_features_base,
        'interaction_features': interaction_features,
        'selected_features': selected_features,
        'user_history_train': user_history_train,
        'user_history_test': user_history_test,
        'als_cache': als_cache,
        'popular_items_idx': popular_items_idx,
        'user_map': user_map,
        'item_map': item_map,
        'ALL_ITEMS': ALL_ITEMS
    }

    def objective(trial):
        # Paramètres LightGBM proposés par Optuna
        params = {
            'objective': 'lambdarank',
            'metric': 'map',
            'eval_at': [12],
            'random_state': 42,
            'n_jobs': -1,
            'force_row_wise': True,
            'verbose': -1,

            'num_leaves': trial.suggest_int('num_leaves', 31, 255),
            'max_depth': trial.suggest_int('max_depth', 6, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 200, 700, step=100),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.5)
        }
        
        # Ajustement sécurité : num_leaves < 2^max_depth
        if params['num_leaves'] >= 2 ** params['max_depth']:
            params['num_leaves'] = 2 ** params['max_depth'] - 1

        # Évaluation du pipeline avec ces paramètres
        map12, _, _, _ = evaluate_model_pipeline(
            lgbm_params=params, **global_data
        )
        return map12

    # Création du study Optuna
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        study_name='h&m_als_temporal'
    )

    start = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    # Résultats
    print("\n" + "=" * 70)
    print("RESULTATS OPTIMISATION")
    print("=" * 70)
    print(f"Temps écoulé: {(time.time()-start)/60:.1f} min")
    print(f"Meilleur MAP@12: {study.best_value:.6f}")
    
    print("\nMeilleurs hyperparamètres:")
    for key, value in study.best_params.items():
        print(f"  {key:20s}: {value}")

    # Modèle final avec les meilleurs paramètres
    best_params = {
        'objective': 'lambdarank',
        'metric': 'map',
        'eval_at': [12],
        'random_state': 42,
        'n_jobs': -1,
        'force_row_wise': True,
        'verbose': -1,
        **study.best_params
    }

    best_map12, best_ranker, _, _ = evaluate_model_pipeline(
        lgbm_params=best_params, **global_data
    )

    # Sauvegarde modèle final
    joblib.dump(best_ranker, 'lightgbm_ranker_TUNED_ALS_temporal.pkl')
    print("\n[SAVE] lightgbm_ranker_TUNED_ALS_temporal.pkl")

    return study, best_ranker, best_map12


In [ ]:
def evaluate_model_pipeline(
    lgbm_params,
    transactions_train,
    transactions_test,
    user_features_base,
    item_features_base,
    interaction_features,
    selected_features,
    user_history_train,
    user_history_test,
    als_cache,
    popular_items_idx,
    user_map,
    item_map,
    ALL_ITEMS
):
    """
    Entraîne un LGBMRanker avec les hyperparamètres fournis et retourne :
    map12, modèle entraîné, X_df avec scores, dict prédictions
    """
    # 1. Construire le dataset complet pour le training
    X_df = build_dataset_candidates(
        users=list(user_history_train.keys()),
        candidate_gen=CandidateGenerator(
            als_cache=als_cache,
            popular_items_idx=popular_items_idx,
            user_history_train=user_history_train,
            item_map=item_map,
            ALL_ITEMS=ALL_ITEMS,
            n_als=20,
            n_popular=15,
            n_repurchase=3
        ),
        user_features=user_features_base,
        item_features=item_features_base,
        interaction_features=interaction_features,
        user_history_test=user_history_test,
        selected_features=selected_features
    )
    
    # 2. Préparer X et y
    y = X_df['label'].values
    X = X_df[selected_features].copy()
    group = X_df.groupby('user_id', observed=True).size().values

    # 3. Entraîner le ranker
    ranker = lgb.LGBMRanker(**lgbm_params)
    ranker.fit(X, y, group=group)

    # 4. Prédiction
    X_df['pred_score'] = ranker.predict(X)
    X_df.sort_values(['user_id', 'pred_score'], ascending=[True, False], inplace=True)
    preds_dict = (
        X_df.groupby('user_id', observed=True)['item_id']
        .head(12)
        .groupby(X_df['user_id'])
        .apply(list)
        .to_dict()
    )

    # 5. Calcul MAP@12
    map12 = map_at_k_fast(preds_dict, user_history_test, k=12)

    return map12, ranker, X_df, preds_dict


In [ ]:
# =========================================================
# APPEL DE L'OPTIMISATION
# =========================================================
n_trials = 5  # ou 30, selon ton temps disponible

study, best_ranker, best_map12 = optimize_with_optuna(
    transactions_train=transactions_train,
    transactions_test=transactions_test,
    user_features_base=user_features,
    item_features_base=item_features,
    interaction_features=interaction_features,
    selected_features=selected_features,
    user_history_train=user_history_train,
    user_history_test=user_history_test,
    als_cache=als_cache,
    popular_items_idx=popular_items_idx,
    user_map=user_map,
    item_map=item_map,
    ALL_ITEMS=ALL_ITEMS,
    n_trials=n_trials
)

print(f"\n Optimisation terminée - Meilleur MAP@12: {best_map12:.6f}")


[I 2026-01-20 23:45:55,949] A new study created in memory with name: h&m_als_temporal


OPTIMISATION OPTUNA - 5 TRIALS


  0%|          | 0/5 [00:00<?, ?it/s]

Build dataset candidates: 100%|██████████| 274594/274594 [27:41<00:00, 165.30it/s]


[W 2026-01-21 00:14:51,546] Trial 0 failed with parameters: {'num_leaves': 115, 'max_depth': 12, 'learning_rate': 0.05395030966670229, 'n_estimators': 500, 'min_child_samples': 24, 'reg_alpha': 0.15599452033620265, 'reg_lambda': 0.05808361216819946, 'subsample': 0.9464704583099741, 'colsample_bytree': 0.8404460046972835, 'min_gain_to_split': 0.35403628889802274} because of the following error: KeyError("['user_club_member', 'user_diversity_rate', 'user_news_regular', 'user_age_group', 'item_product_type_no', 'user_purchase_frequency', 'user_n_purchases', 'user_recency_days', 'item_sales_last_4w', 'item_n_sales', 'item_sales_last_1w', 'item_trend_4w', 'item_sales_last_2w', 'item_repurchase_rate', 'item_trend_1w', 'user_avg_price'] not in index").
Traceback (most recent call last):
  File "C:\Users\ajarr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_

KeyError: "['user_club_member', 'user_diversity_rate', 'user_news_regular', 'user_age_group', 'item_product_type_no', 'user_purchase_frequency', 'user_n_purchases', 'user_recency_days', 'item_sales_last_4w', 'item_n_sales', 'item_sales_last_1w', 'item_trend_4w', 'item_sales_last_2w', 'item_repurchase_rate', 'item_trend_1w', 'user_avg_price'] not in index"